# Feature Engineering
Step 1.1: Create Monthly Base Dataset
Objectives

- Aggregate transactions to monthly granularity
- Create complete time index for all branches
- Handle missing periods appropriately

Step 1.2: Weather Feature Engineering
Objectives

- Collect and process weather data
- Calculate HVAC-specific weather features (CDD, HDD)
- Create lagged weather features

Step 1.3: External Data Collection
Objectives

- Add Google Trends data

Step 1.4: Lag & Rolling Features
Objectives

- Create lag features (1, 3, 6, 12 months)
- Create rolling statistics (mean, std, growth rates)
- Create interaction features

In [2]:
import pandas as pd

In [3]:
main_df = pd.read_csv('../data/refactored_df.csv')
weather_df = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

In [4]:
# Step 1: Data Preparation and Merging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
print(f"Main dataset shape: {main_df.shape}")
print(f"Weather dataset shape: {weather_df.shape}")
print(f"Trends dataset shape: {trends_df.shape}")

# Convert date columns
main_df['Date'] = pd.to_datetime(main_df['Date'])
weather_df['Date'] = pd.to_datetime(weather_df['Date'])
trends_df['Month'] = pd.to_datetime(trends_df['Month'])

print("\nDate ranges:")
print(f"Main data: {main_df['Date'].min()} to {main_df['Date'].max()}")
print(f"Weather data: {weather_df['Date'].min()} to {weather_df['Date'].max()}")
print(f"Trends data: {trends_df['Month'].min()} to {trends_df['Month'].max()}")


Loading datasets...
Main dataset shape: (147595, 9)
Weather dataset shape: (456, 11)
Trends dataset shape: (82, 2)

Date ranges:
Main data: 2019-04-03 00:00:00 to 2024-03-30 00:00:00
Weather data: 2019-04-01 00:00:00 to 2025-10-01 00:00:00
Trends data: 2019-01-01 00:00:00 to 2025-10-01 00:00:00


In [6]:
# Step 2: Create Monthly Base Dataset
print("Creating monthly aggregated dataset...")

# Aggregate to monthly level by branch and segment
monthly_df = main_df.groupby(['Year', 'Month', 'Branch', 'Segment', 'Rating', 'Tonnage']).agg({
    'Qty': 'sum',
    'Date': 'first'
}).reset_index()

# Create complete time index for all combinations
date_range = pd.date_range(start='2019-04-01', end='2024-03-31', freq='MS')  # Month start
branches = main_df['Branch'].unique()
segments = main_df['Segment'].unique()
ratings = main_df['Rating'].unique()
tonnages = main_df['Tonnage'].unique()

# Create complete index
complete_index = []
for date in date_range:
    for branch in branches:
        for segment in segments:
            for rating in ratings:
                for tonnage in tonnages:
                    complete_index.append({
                        'Date': date,
                        'Year': date.year,
                        'Month': date.month,
                        'Branch': branch,
                        'Segment': segment,
                        'Rating': rating,
                        'Tonnage': tonnage
                    })

complete_df = pd.DataFrame(complete_index)

# Merge with actual data
monthly_df = complete_df.merge(
    monthly_df, 
    on=['Year', 'Month', 'Branch', 'Segment', 'Rating', 'Tonnage'], 
    how='left'
)

# Fill missing quantities with 0
monthly_df['Qty'] = monthly_df['Qty'].fillna(0)

print(f"Monthly dataset shape: {monthly_df.shape}")
# print(f"Date range: {monthly_df['Date'].min()} to {monthly_df['Date'].max()}")
print(f"Branches: {sorted(monthly_df['Branch'].unique())}")
print(f"Segments: {sorted(monthly_df['Segment'].unique())}")
print(f"Total records: {len(monthly_df)}")
print(f"Non-zero records: {len(monthly_df[monthly_df['Qty'] > 0])}")


Creating monthly aggregated dataset...


Monthly dataset shape: (15000, 9)
Branches: ['BLR', 'COK', 'MAA', 'SBD', 'SBD1']
Segments: ['Inverter', 'Non Inv']
Total records: 15000
Non-zero records: 3739


In [7]:
monthly_df

,Date_x,Year,Month,Branch,Segment,Rating,Tonnage,Qty,Date_y
0,2019-04-01,2019,4,MAA,Inverter,4 Star,2.0,119.0,2019-04-03
1,2019-04-01,2019,4,MAA,Inverter,4 Star,1.5,107.0,2019-04-05
2,2019-04-01,2019,4,MAA,Inverter,4 Star,1.0,15.0,2019-04-05
3,2019-04-01,2019,4,MAA,Inverter,4 Star,1.8,8.0,2019-04-11
4,2019-04-01,2019,4,MAA,Inverter,4 Star,0.8,0.0,NaT
...,...,...,...,...,...,...,...,...,...
14995,2024-03-01,2024,3,COK,Non Inv,1 Star,2.0,0.0,NaT
14996,2024-03-01,2024,3,COK,Non Inv,1 Star,1.5,0.0,NaT
14997,2024-03-01,2024,3,COK,Non Inv,1 Star,1.0,0.0,NaT
14998,2024-03-01,2024,3,COK,Non Inv,1 Star,1.8,136.0,2024-03-04


In [8]:
# Step 3: Temporal Features Engineering
print("Creating temporal features...")

# Extract temporal components
monthly_df['year'] = monthly_df['Date'].dt.year
monthly_df['month'] = monthly_df['Date'].dt.month
monthly_df['quarter'] = monthly_df['Date'].dt.quarter
monthly_df['day_of_year'] = monthly_df['Date'].dt.dayofyear
monthly_df['days_in_month'] = monthly_df['Date'].dt.days_in_month

# Cyclical encoding for seasonality (as recommended in EDA)
monthly_df['month_sin'] = np.sin(2 * np.pi * monthly_df['month'] / 12)
monthly_df['month_cos'] = np.cos(2 * np.pi * monthly_df['month'] / 12)
monthly_df['quarter_sin'] = np.sin(2 * np.pi * monthly_df['quarter'] / 4)
monthly_df['quarter_cos'] = np.cos(2 * np.pi * monthly_df['quarter'] / 4)
monthly_df['day_of_year_sin'] = np.sin(2 * np.pi * monthly_df['day_of_year'] / 365)
monthly_df['day_of_year_cos'] = np.cos(2 * np.pi * monthly_df['day_of_year'] / 365)

# Seasonal indicators based on EDA findings
# Peak months: March, December, February
# Low months: November, August, July
monthly_df['is_peak_month'] = monthly_df['month'].isin([3, 12, 2]).astype(int)
monthly_df['is_low_month'] = monthly_df['month'].isin([11, 8, 7]).astype(int)
monthly_df['is_summer'] = monthly_df['month'].isin([6, 7, 8]).astype(int)
monthly_df['is_winter'] = monthly_df['month'].isin([12, 1, 2]).astype(int)
monthly_df['is_monsoon'] = monthly_df['month'].isin([9, 10, 11]).astype(int)

# Quarterly indicators
monthly_df['is_q1'] = (monthly_df['quarter'] == 1).astype(int)
monthly_df['is_q2'] = (monthly_df['quarter'] == 2).astype(int)
monthly_df['is_q3'] = (monthly_df['quarter'] == 3).astype(int)
monthly_df['is_q4'] = (monthly_df['quarter'] == 4).astype(int)

# Time trend
monthly_df['time_trend'] = (monthly_df['Date'] - monthly_df['Date'].min()).dt.days

print("Temporal features created:")
print(f"- Cyclical features: month_sin/cos, quarter_sin/cos, day_of_year_sin/cos")
print(f"- Seasonal indicators: peak_month, low_month, summer, winter, monsoon")
print(f"- Quarterly indicators: q1-q4")
print(f"- Time trend: {monthly_df['time_trend'].min()} to {monthly_df['time_trend'].max()} days")


Creating temporal features...


KeyError: 'Date'

In [ ]:
# Step 4: Lag Features Engineering
print("Creating lag features...")

# Sort by branch, segment, rating, tonnage, and date for proper lag calculation
monthly_df = monthly_df.sort_values(['Branch', 'Segment', 'Rating', 'Tonnage', 'Date']).reset_index(drop=True)

# Create lag features as recommended in EDA (1, 7, 14, 30, 90, 365 days equivalent in months)
# For monthly data, we'll use: 1, 3, 6, 12 months (equivalent to daily lags)
lag_periods = [1, 3, 6, 12]  # months

for lag in lag_periods:
    monthly_df[f'qty_lag_{lag}m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].shift(lag)

# Year-over-year features (lag-12 is yearly seasonality)
monthly_df['qty_yoy'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].shift(12)

# Create lag features for different aggregations
# Branch-level lags
branch_monthly = monthly_df.groupby(['Branch', 'Date'])['Qty'].sum().reset_index()
branch_monthly = branch_monthly.sort_values(['Branch', 'Date']).reset_index(drop=True)

for lag in lag_periods:
    branch_monthly[f'branch_qty_lag_{lag}m'] = branch_monthly.groupby('Branch')['Qty'].shift(lag)

# Merge branch-level lags back
monthly_df = monthly_df.merge(
    branch_monthly[['Branch', 'Date'] + [f'branch_qty_lag_{lag}m' for lag in lag_periods]], 
    on=['Branch', 'Date'], 
    how='left'
)

# Segment-level lags
segment_monthly = monthly_df.groupby(['Segment', 'Date'])['Qty'].sum().reset_index()
segment_monthly = segment_monthly.sort_values(['Segment', 'Date']).reset_index(drop=True)

for lag in lag_periods:
    segment_monthly[f'segment_qty_lag_{lag}m'] = segment_monthly.groupby('Segment')['Qty'].shift(lag)

# Merge segment-level lags back
monthly_df = monthly_df.merge(
    segment_monthly[['Segment', 'Date'] + [f'segment_qty_lag_{lag}m' for lag in lag_periods]], 
    on=['Segment', 'Date'], 
    how='left'
)

print("Lag features created:")
print(f"- Product-level lags: {[f'qty_lag_{lag}m' for lag in lag_periods]}")
print(f"- Year-over-year: qty_yoy")
print(f"- Branch-level lags: {[f'branch_qty_lag_{lag}m' for lag in lag_periods]}")
print(f"- Segment-level lags: {[f'segment_qty_lag_{lag}m' for lag in lag_periods]}")


In [ ]:
# Step 5: Rolling Statistics and Moving Averages
print("Creating rolling statistics...")

# Rolling windows (in months, equivalent to daily windows)
rolling_windows = [3, 6, 12]  # 3, 6, 12 months

# Product-level rolling statistics
for window in rolling_windows:
    monthly_df[f'qty_ma_{window}m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    monthly_df[f'qty_std_{window}m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].rolling(window, min_periods=1).std().reset_index(0, drop=True)
    monthly_df[f'qty_min_{window}m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].rolling(window, min_periods=1).min().reset_index(0, drop=True)
    monthly_df[f'qty_max_{window}m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].rolling(window, min_periods=1).max().reset_index(0, drop=True)

# Growth rates
monthly_df['qty_growth_1m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].pct_change(1)
monthly_df['qty_growth_3m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].pct_change(3)
monthly_df['qty_growth_12m'] = monthly_df.groupby(['Branch', 'Segment', 'Rating', 'Tonnage'])['Qty'].pct_change(12)

# Branch-level rolling statistics
branch_monthly = monthly_df.groupby(['Branch', 'Date'])['Qty'].sum().reset_index()
branch_monthly = branch_monthly.sort_values(['Branch', 'Date']).reset_index(drop=True)

for window in rolling_windows:
    branch_monthly[f'branch_qty_ma_{window}m'] = branch_monthly.groupby('Branch')['Qty'].rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    branch_monthly[f'branch_qty_std_{window}m'] = branch_monthly.groupby('Branch')['Qty'].rolling(window, min_periods=1).std().reset_index(0, drop=True)

# Merge branch rolling stats back
monthly_df = monthly_df.merge(
    branch_monthly[['Branch', 'Date'] + [f'branch_qty_ma_{window}m' for window in rolling_windows] + 
                   [f'branch_qty_std_{window}m' for window in rolling_windows]], 
    on=['Branch', 'Date'], 
    how='left'
)

# Segment-level rolling statistics
segment_monthly = monthly_df.groupby(['Segment', 'Date'])['Qty'].sum().reset_index()
segment_monthly = segment_monthly.sort_values(['Segment', 'Date']).reset_index(drop=True)

for window in rolling_windows:
    segment_monthly[f'segment_qty_ma_{window}m'] = segment_monthly.groupby('Segment')['Qty'].rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    segment_monthly[f'segment_qty_std_{window}m'] = segment_monthly.groupby('Segment')['Qty'].rolling(window, min_periods=1).std().reset_index(0, drop=True)

# Merge segment rolling stats back
monthly_df = monthly_df.merge(
    segment_monthly[['Segment', 'Date'] + [f'segment_qty_ma_{window}m' for window in rolling_windows] + 
                    [f'segment_qty_std_{window}m' for window in rolling_windows]], 
    on=['Segment', 'Date'], 
    how='left'
)

print("Rolling statistics created:")
print(f"- Product-level: MA, STD, MIN, MAX for windows {rolling_windows}")
print(f"- Growth rates: 1m, 3m, 12m")
print(f"- Branch-level: MA, STD for windows {rolling_windows}")
print(f"- Segment-level: MA, STD for windows {rolling_windows}")


In [ ]:
# Step 6: Weather Features Engineering
print("Creating weather features...")

# Prepare weather data for merging
weather_df['year'] = weather_df['Date'].dt.year
weather_df['month'] = weather_df['Date'].dt.month

# Create monthly weather aggregations
weather_monthly = weather_df.groupby(['Branch', 'year', 'month']).agg({
    'Min Temp': 'min',
    'Max Temp': 'max', 
    'Avg Temp': 'mean',
    'Min Humidity': 'min',
    'Max Humidity': 'max',
    'Avg Humidity': 'mean',
    'Min Wind Speed': 'min',
    'Max Wind Speed': 'max',
    'Avg Wind Speed': 'mean'
}).reset_index()

# Create date column for merging
weather_monthly['Date'] = pd.to_datetime(weather_monthly[['year', 'month']].assign(day=1))

# Merge weather data
monthly_df = monthly_df.merge(
    weather_monthly[['Branch', 'Date', 'Min Temp', 'Max Temp', 'Avg Temp', 
                     'Min Humidity', 'Max Humidity', 'Avg Humidity',
                     'Min Wind Speed', 'Max Wind Speed', 'Avg Wind Speed']], 
    on=['Branch', 'Date'], 
    how='left'
)

# Create weather features with 3-month lead (as recommended in EDA)
weather_lead_cols = ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Max Temp', 'Min Temp']

for col in weather_lead_cols:
    monthly_df[f'{col.lower().replace(" ", "_")}_lead_3m'] = monthly_df.groupby('Branch')[col].shift(-3)

# Weather extremes based on EDA findings
# Temperature extremes
temp_80th = monthly_df['Max Temp'].quantile(0.8)
temp_20th = monthly_df['Min Temp'].quantile(0.2)
monthly_df['hot_day'] = (monthly_df['Max Temp'] > temp_80th).astype(int)
monthly_df['cold_day'] = (monthly_df['Min Temp'] < temp_20th).astype(int)

# Humidity extremes
humidity_80th = monthly_df['Avg Humidity'].quantile(0.8)
humidity_20th = monthly_df['Avg Humidity'].quantile(0.2)
monthly_df['humid_day'] = (monthly_df['Avg Humidity'] > humidity_80th).astype(int)
monthly_df['dry_day'] = (monthly_df['Avg Humidity'] < humidity_20th).astype(int)

# Temperature range
monthly_df['temp_range'] = monthly_df['Max Temp'] - monthly_df['Min Temp']

# HVAC-specific weather features
# Cooling Degree Days (CDD) - when temperature > 24°C
monthly_df['cdd'] = np.maximum(monthly_df['Avg Temp'] - 24, 0) * monthly_df['days_in_month']

# Heating Degree Days (HDD) - when temperature < 18°C  
monthly_df['hdd'] = np.maximum(18 - monthly_df['Avg Temp'], 0) * monthly_df['days_in_month']

# Comfort index (temperature and humidity combined)
monthly_df['comfort_index'] = monthly_df['Avg Temp'] - (monthly_df['Avg Humidity'] / 10)

# Weather lag features (1, 3, 6 months)
weather_lag_cols = ['Avg Temp', 'Avg Humidity', 'Max Temp', 'cdd']
for col in weather_lag_cols:
    for lag in [1, 3, 6]:
        monthly_df[f'{col.lower().replace(" ", "_")}_lag_{lag}m'] = monthly_df.groupby('Branch')[col].shift(lag)

print("Weather features created:")
print(f"- Basic weather: temp, humidity, wind speed (min, max, avg)")
print(f"- Weather extremes: hot_day, cold_day, humid_day, dry_day")
print(f"- HVAC features: cdd, hdd, comfort_index, temp_range")
print("- Weather with 3-month lead: avg_temp_lead_3m, avg_humidity_lead_3m, etc.")
print("- Weather lags: avg_temp_lag_1m, avg_temp_lag_3m, avg_temp_lag_6m, etc.")


In [ ]:
# Step 7: Product Evolution and Segment Features
print("Creating product evolution features...")

# Segment evolution features (as recommended in EDA)
# Inverter penetration rate by branch and time
inverter_penetration = monthly_df.groupby(['Branch', 'Date']).apply(
    lambda x: (x['Segment'] == 'Inverter').mean()
).reset_index(name='inverter_penetration')

monthly_df = monthly_df.merge(inverter_penetration, on=['Branch', 'Date'], how='left')

# Tonnage mix features
tonnage_1_5_share = monthly_df.groupby(['Branch', 'Date']).apply(
    lambda x: (x['Tonnage'] == 1.5).mean()
).reset_index(name='tonnage_1_5_share')

monthly_df = monthly_df.merge(tonnage_1_5_share, on=['Branch', 'Date'], how='left')

# Star rating evolution
star_5_share = monthly_df.groupby(['Branch', 'Date']).apply(
    lambda x: (x['Rating'] == '5 Star').mean()
).reset_index(name='star_5_share')

monthly_df = monthly_df.merge(star_5_share, on=['Branch', 'Date'], how='left')

# Product lifecycle features
# Age of product category (years since first appearance)
product_first_appearance = monthly_df.groupby(['Segment', 'Rating', 'Tonnage'])['Date'].min().reset_index()
product_first_appearance.columns = ['Segment', 'Rating', 'Tonnage', 'first_appearance']

monthly_df = monthly_df.merge(product_first_appearance, on=['Segment', 'Rating', 'Tonnage'], how='left')
monthly_df['product_age_months'] = ((monthly_df['Date'] - monthly_df['first_appearance']).dt.days / 30).round()

# Technology adoption curve features
# Inverter adoption rate by branch over time
inverter_adoption = monthly_df.groupby(['Branch', 'Date'])['inverter_penetration'].first().reset_index()
inverter_adoption = inverter_adoption.sort_values(['Branch', 'Date']).reset_index(drop=True)

# Calculate adoption velocity (rate of change)
inverter_adoption['adoption_velocity'] = inverter_adoption.groupby('Branch')['inverter_penetration'].diff()

# Merge back
monthly_df = monthly_df.merge(
    inverter_adoption[['Branch', 'Date', 'adoption_velocity']], 
    on=['Branch', 'Date'], 
    how='left'
)

# Capacity mix features
capacity_mix = monthly_df.groupby(['Branch', 'Date']).agg({
    'Tonnage': lambda x: x.value_counts().to_dict()
}).reset_index()

# Create tonnage distribution features
tonnage_categories = [0.8, 1.0, 1.5, 2.0]
for tonnage in tonnage_categories:
    monthly_df[f'tonnage_{tonnage}_share'] = monthly_df.groupby(['Branch', 'Date']).apply(
        lambda x: (x['Tonnage'] == tonnage).mean()
    ).reset_index(0, drop=True)

# Segment performance ratios
segment_performance = monthly_df.groupby(['Branch', 'Date', 'Segment'])['Qty'].sum().reset_index()
segment_performance_pivot = segment_performance.pivot_table(
    index=['Branch', 'Date'], 
    columns='Segment', 
    values='Qty', 
    fill_value=0
).reset_index()

# Calculate inverter vs non-inverter ratio
segment_performance_pivot['inverter_ratio'] = (
    segment_performance_pivot['Inverter'] / 
    (segment_performance_pivot['Inverter'] + segment_performance_pivot['Non Inv'] + 1e-8)
)

monthly_df = monthly_df.merge(
    segment_performance_pivot[['Branch', 'Date', 'inverter_ratio']], 
    on=['Branch', 'Date'], 
    how='left'
)

print("Product evolution features created:")
print("- inverter_penetration: Inverter market share by branch/time")
print("- tonnage_1_5_share: 1.5T tonnage share by branch/time")
print("- star_5_share: 5-star rating share by branch/time")
print("- product_age_months: Age of product category")
print("- adoption_velocity: Rate of inverter adoption change")
print("- tonnage_X_share: Share of each tonnage category")
print("- inverter_ratio: Inverter vs non-inverter performance ratio")


In [ ]:
# Step 8: Branch-specific and Geographic Features
print("Creating geographic features...")

# Branch-specific seasonality patterns (as recommended in EDA)
branch_seasonality = monthly_df.groupby(['Branch', 'month'])['Qty'].mean().reset_index()
branch_seasonality_pivot = branch_seasonality.pivot_table(
    index='Branch', 
    columns='month', 
    values='Qty', 
    fill_value=0
).reset_index()

# Calculate branch-specific seasonal strength
branch_seasonal_strength = {}
for branch in monthly_df['Branch'].unique():
    branch_data = monthly_df[monthly_df['Branch'] == branch]
    monthly_avg = branch_data.groupby('month')['Qty'].mean()
    seasonal_strength = monthly_avg.std() / monthly_avg.mean()
    branch_seasonal_strength[branch] = seasonal_strength

# Create branch seasonal strength feature
monthly_df['branch_seasonal_strength'] = monthly_df['Branch'].map(branch_seasonal_strength)

# Market share evolution by branch
total_monthly_qty = monthly_df.groupby('Date')['Qty'].sum().reset_index()
total_monthly_qty.columns = ['Date', 'total_qty']

monthly_df = monthly_df.merge(total_monthly_qty, on='Date', how='left')

# Branch market share
branch_monthly_qty = monthly_df.groupby(['Branch', 'Date'])['Qty'].sum().reset_index()
branch_monthly_qty.columns = ['Branch', 'Date', 'branch_qty']

monthly_df = monthly_df.merge(branch_monthly_qty, on=['Branch', 'Date'], how='left')
monthly_df['branch_market_share'] = monthly_df['branch_qty'] / (monthly_df['total_qty'] + 1e-8)

# Branch performance relative to average
monthly_df['branch_performance_ratio'] = monthly_df['branch_qty'] / (
    monthly_df.groupby('Date')['branch_qty'].transform('mean') + 1e-8
)

# Regional preferences and competitive dynamics
# Branch growth rates
branch_growth = monthly_df.groupby(['Branch', 'Date'])['Qty'].sum().reset_index()
branch_growth = branch_growth.sort_values(['Branch', 'Date']).reset_index(drop=True)

# Calculate year-over-year growth
branch_growth['qty_yoy'] = branch_growth.groupby('Branch')['Qty'].pct_change(12)
branch_growth['qty_3m_growth'] = branch_growth.groupby('Branch')['Qty'].pct_change(3)

# Merge growth features back
monthly_df = monthly_df.merge(
    branch_growth[['Branch', 'Date', 'qty_yoy', 'qty_3m_growth']], 
    on=['Branch', 'Date'], 
    how='left'
)

# Branch clustering based on EDA findings
# High growth: COK, SBD
# Declining: MAA, SBD1
# Stable: BLR
branch_clusters = {
    'COK': 'high_growth',
    'SBD': 'high_growth', 
    'MAA': 'declining',
    'SBD1': 'declining',
    'BLR': 'stable'
}

monthly_df['branch_cluster'] = monthly_df['Branch'].map(branch_clusters)

# Branch-specific weather sensitivity
# Calculate correlation between weather and sales by branch
branch_weather_sensitivity = {}
for branch in monthly_df['Branch'].unique():
    branch_data = monthly_df[monthly_df['Branch'] == branch]
    if len(branch_data) > 10:  # Minimum data points
        temp_corr = branch_data['Avg Temp'].corr(branch_data['Qty'])
        humidity_corr = branch_data['Avg Humidity'].corr(branch_data['Qty'])
        branch_weather_sensitivity[branch] = {
            'temp_sensitivity': temp_corr if not pd.isna(temp_corr) else 0,
            'humidity_sensitivity': humidity_corr if not pd.isna(humidity_corr) else 0
        }
    else:
        branch_weather_sensitivity[branch] = {'temp_sensitivity': 0, 'humidity_sensitivity': 0}

# Create weather sensitivity features
monthly_df['branch_temp_sensitivity'] = monthly_df['Branch'].map(
    lambda x: branch_weather_sensitivity[x]['temp_sensitivity']
)
monthly_df['branch_humidity_sensitivity'] = monthly_df['Branch'].map(
    lambda x: branch_weather_sensitivity[x]['humidity_sensitivity']
)

# Branch-specific product preferences
branch_segment_pref = monthly_df.groupby(['Branch', 'Segment'])['Qty'].sum().reset_index()
branch_segment_pref_pivot = branch_segment_pref.pivot_table(
    index='Branch', 
    columns='Segment', 
    values='Qty', 
    fill_value=0
).reset_index()

# Calculate segment preference ratios
branch_segment_pref_pivot['inverter_preference'] = (
    branch_segment_pref_pivot['Inverter'] / 
    (branch_segment_pref_pivot['Inverter'] + branch_segment_pref_pivot['Non Inv'] + 1e-8)
)

monthly_df = monthly_df.merge(
    branch_segment_pref_pivot[['Branch', 'inverter_preference']], 
    on='Branch', 
    how='left'
)

print("Geographic features created:")
print("- branch_seasonal_strength: Seasonal variation strength by branch")
print("- branch_market_share: Market share evolution by branch")
print("- branch_performance_ratio: Performance relative to average")
print("- branch_cluster: High growth/declining/stable classification")
print("- branch_temp_sensitivity: Temperature correlation by branch")
print("- branch_humidity_sensitivity: Humidity correlation by branch")
print("- inverter_preference: Branch preference for inverter ACs")


In [ ]:
# Step 9: Customer Trends Features
print("Creating customer trends features...")

# Prepare trends data for merging
trends_df['year'] = trends_df['Month'].dt.year
trends_df['month'] = trends_df['Month'].dt.month
trends_df['Date'] = pd.to_datetime(trends_df[['year', 'month']].assign(day=1))

# Merge customer trends data
monthly_df = monthly_df.merge(
    trends_df[['Date', 'Interest']], 
    on='Date', 
    how='left'
)

# Customer interest with 3-month lead (as recommended in EDA)
monthly_df['interest_lead_3m'] = monthly_df['Interest'].shift(-3)

# Interest categories based on EDA findings
interest_33rd = monthly_df['Interest'].quantile(0.33)
interest_67th = monthly_df['Interest'].quantile(0.67)

monthly_df['interest_level'] = pd.cut(
    monthly_df['Interest'], 
    bins=[0, interest_33rd, interest_67th, 100], 
    labels=['Low', 'Medium', 'High']
)

# Interest level indicators
monthly_df['is_high_interest'] = (monthly_df['Interest'] > interest_67th).astype(int)
monthly_df['is_low_interest'] = (monthly_df['Interest'] < interest_33rd).astype(int)

# Interest momentum features
monthly_df['interest_momentum'] = monthly_df['Interest'].diff()
monthly_df['interest_acceleration'] = monthly_df['interest_momentum'].diff()

# Interest lag features
monthly_df['interest_lag_1m'] = monthly_df['Interest'].shift(1)
monthly_df['interest_lag_3m'] = monthly_df['Interest'].shift(3)
monthly_df['interest_lag_6m'] = monthly_df['Interest'].shift(6)

# Interest rolling statistics
monthly_df['interest_ma_3m'] = monthly_df['Interest'].rolling(3, min_periods=1).mean()
monthly_df['interest_ma_6m'] = monthly_df['Interest'].rolling(6, min_periods=1).mean()
monthly_df['interest_std_3m'] = monthly_df['Interest'].rolling(3, min_periods=1).std()

# Seasonal interest patterns
interest_seasonal = monthly_df.groupby('month')['Interest'].mean()
monthly_df['interest_seasonal_avg'] = monthly_df['month'].map(interest_seasonal)
monthly_df['interest_vs_seasonal'] = monthly_df['Interest'] - monthly_df['interest_seasonal_avg']

# Interest-sales interaction features
# Calculate correlation between interest and sales
monthly_df['interest_sales_correlation'] = monthly_df['Interest'].rolling(12, min_periods=6).corr(monthly_df['Qty'])

# Interest impact features (based on EDA: 191 units change per interest point)
monthly_df['interest_impact'] = monthly_df['Interest'] * 191  # Units per interest point
monthly_df['interest_impact_lead_3m'] = monthly_df['interest_impact'].shift(-3)

print("Customer trends features created:")
print("- Interest: Raw customer interest data")
print("- interest_lead_3m: Interest with 3-month lead")
print("- interest_level: Low/Medium/High categories")
print("- is_high_interest/is_low_interest: Binary indicators")
print("- interest_momentum: Rate of change in interest")
print("- interest_acceleration: Second derivative of interest")
print("- interest_lag_Xm: Interest lags (1, 3, 6 months)")
print("- interest_ma_Xm: Interest moving averages (3, 6 months)")
print("- interest_seasonal_avg: Seasonal average interest")
print("- interest_vs_seasonal: Deviation from seasonal average")
print("- interest_sales_correlation: Rolling correlation with sales")
print("- interest_impact: Estimated sales impact (191 units per point)")
print("- interest_impact_lead_3m: Impact with 3-month lead")


In [ ]:
# Step 10: Interaction Features and Transformations
print("Creating interaction features and transformations...")

# Weather-Product interactions
monthly_df['temp_inverter_interaction'] = monthly_df['Avg Temp'] * monthly_df['inverter_penetration']
monthly_df['humidity_tonnage_interaction'] = monthly_df['Avg Humidity'] * monthly_df['Tonnage']
monthly_df['cdd_segment_interaction'] = monthly_df['cdd'] * (monthly_df['Segment'] == 'Inverter').astype(int)

# Weather-Season interactions
monthly_df['temp_peak_interaction'] = monthly_df['Avg Temp'] * monthly_df['is_peak_month']
monthly_df['humidity_low_interaction'] = monthly_df['Avg Humidity'] * monthly_df['is_low_month']
monthly_df['cdd_summer_interaction'] = monthly_df['cdd'] * monthly_df['is_summer']

# Branch-Product interactions
monthly_df['branch_inverter_interaction'] = monthly_df['branch_market_share'] * monthly_df['inverter_penetration']
monthly_df['branch_tonnage_interaction'] = monthly_df['branch_market_share'] * monthly_df['Tonnage']

# Interest-Weather interactions
monthly_df['interest_temp_interaction'] = monthly_df['Interest'] * monthly_df['Avg Temp']
monthly_df['interest_humidity_interaction'] = monthly_df['Interest'] * monthly_df['Avg Humidity']
monthly_df['interest_cdd_interaction'] = monthly_df['Interest'] * monthly_df['cdd']

# Interest-Product interactions
monthly_df['interest_inverter_interaction'] = monthly_df['Interest'] * monthly_df['inverter_penetration']
monthly_df['interest_tonnage_interaction'] = monthly_df['Interest'] * monthly_df['Tonnage']

# Branch-Weather interactions
monthly_df['branch_temp_interaction'] = monthly_df['branch_temp_sensitivity'] * monthly_df['Avg Temp']
monthly_df['branch_humidity_interaction'] = monthly_df['branch_humidity_sensitivity'] * monthly_df['Avg Humidity']

# Log transformations for skewed variables (as recommended in EDA)
monthly_df['log_qty'] = np.log1p(monthly_df['Qty'])  # log(1 + qty) to handle zeros
monthly_df['log_branch_qty'] = np.log1p(monthly_df['branch_qty'])
monthly_df['log_total_qty'] = np.log1p(monthly_df['total_qty'])

# Square root transformations for moderate skewness
monthly_df['sqrt_qty'] = np.sqrt(monthly_df['Qty'])
monthly_df['sqrt_cdd'] = np.sqrt(monthly_df['cdd'])

# Polynomial features for non-linear relationships
monthly_df['temp_squared'] = monthly_df['Avg Temp'] ** 2
monthly_df['humidity_squared'] = monthly_df['Avg Humidity'] ** 2
monthly_df['interest_squared'] = monthly_df['Interest'] ** 2

# Ratio features
monthly_df['temp_humidity_ratio'] = monthly_df['Avg Temp'] / (monthly_df['Avg Humidity'] + 1e-8)
monthly_df['inverter_non_inverter_ratio'] = monthly_df['inverter_penetration'] / (1 - monthly_df['inverter_penetration'] + 1e-8)
monthly_df['peak_low_ratio'] = monthly_df['is_peak_month'] / (monthly_df['is_low_month'] + 1e-8)

# Composite features
monthly_df['weather_comfort_score'] = (
    monthly_df['Avg Temp'] * 0.4 + 
    (100 - monthly_df['Avg Humidity']) * 0.3 + 
    monthly_df['Avg Wind Speed'] * 0.3
)

monthly_df['product_attractiveness'] = (
    monthly_df['inverter_penetration'] * 0.4 + 
    monthly_df['star_5_share'] * 0.3 + 
    monthly_df['tonnage_1_5_share'] * 0.3
)

monthly_df['market_demand_index'] = (
    monthly_df['Interest'] * 0.3 + 
    monthly_df['cdd'] * 0.3 + 
    monthly_df['is_peak_month'] * 0.2 + 
    monthly_df['branch_market_share'] * 0.2
)

# Outlier flags (as recommended in EDA)
qty_q75 = monthly_df['Qty'].quantile(0.75)
qty_q25 = monthly_df['Qty'].quantile(0.25)
qty_iqr = qty_q75 - qty_q25
qty_upper_bound = qty_q75 + 1.5 * qty_iqr

monthly_df['is_qty_outlier'] = (monthly_df['Qty'] > qty_upper_bound).astype(int)

# Temperature outlier flags
temp_q75 = monthly_df['Avg Temp'].quantile(0.75)
temp_q25 = monthly_df['Avg Temp'].quantile(0.25)
temp_iqr = temp_q75 - temp_q25
temp_upper_bound = temp_q75 + 1.5 * temp_iqr
temp_lower_bound = temp_q25 - 1.5 * temp_iqr

monthly_df['is_temp_outlier'] = (
    (monthly_df['Avg Temp'] > temp_upper_bound) | 
    (monthly_df['Avg Temp'] < temp_lower_bound)
).astype(int)

print("Interaction features and transformations created:")
print("- Weather-Product interactions: temp_inverter, humidity_tonnage, cdd_segment")
print("- Weather-Season interactions: temp_peak, humidity_low, cdd_summer")
print("- Branch-Product interactions: branch_inverter, branch_tonnage")
print("- Interest-Weather interactions: interest_temp, interest_humidity, interest_cdd")
print("- Interest-Product interactions: interest_inverter, interest_tonnage")
print("- Branch-Weather interactions: branch_temp, branch_humidity")
print("- Log transformations: log_qty, log_branch_qty, log_total_qty")
print("- Square root transformations: sqrt_qty, sqrt_cdd")
print("- Polynomial features: temp_squared, humidity_squared, interest_squared")
print("- Ratio features: temp_humidity_ratio, inverter_non_inverter_ratio, peak_low_ratio")
print("- Composite features: weather_comfort_score, product_attractiveness, market_demand_index")
print("- Outlier flags: is_qty_outlier, is_temp_outlier")


In [ ]:
# Step 11: Final Dataset Preparation and Summary
print("Preparing final engineered dataset...")

# Handle missing values
print("Handling missing values...")

# Forward fill for weather data
weather_cols = ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Max Temp', 'Min Temp', 
                'Min Humidity', 'Max Humidity', 'Min Wind Speed', 'Max Wind Speed']
for col in weather_cols:
    monthly_df[col] = monthly_df.groupby('Branch')[col].fillna(method='ffill')

# Forward fill for customer trends
monthly_df['Interest'] = monthly_df['Interest'].fillna(method='ffill')
monthly_df['interest_lead_3m'] = monthly_df['interest_lead_3m'].fillna(method='ffill')

# Fill remaining NaN values with 0 for numeric columns
numeric_cols = monthly_df.select_dtypes(include=[np.number]).columns
monthly_df[numeric_cols] = monthly_df[numeric_cols].fillna(0)

# Create categorical encodings
print("Creating categorical encodings...")

# One-hot encode categorical variables
categorical_cols = ['Branch', 'Segment', 'Rating', 'branch_cluster', 'interest_level']
for col in categorical_cols:
    if col in monthly_df.columns:
        dummies = pd.get_dummies(monthly_df[col], prefix=col)
        monthly_df = pd.concat([monthly_df, dummies], axis=1)

# Ordinal encoding for tonnage
tonnage_mapping = {0.8: 1, 1.0: 2, 1.5: 3, 2.0: 4}
monthly_df['tonnage_ordinal'] = monthly_df['Tonnage'].map(tonnage_mapping)

# Create feature importance groups based on EDA recommendations
feature_groups = {
    'temporal': ['month_sin', 'month_cos', 'quarter_sin', 'quarter_cos', 'is_peak_month', 'is_low_month', 
                 'is_summer', 'is_winter', 'is_monsoon', 'is_q1', 'is_q2', 'is_q3', 'is_q4'],
    
    'lag_features': ['qty_lag_1m', 'qty_lag_3m', 'qty_lag_6m', 'qty_lag_12m', 'qty_yoy',
                     'branch_qty_lag_1m', 'branch_qty_lag_3m', 'branch_qty_lag_6m', 'branch_qty_lag_12m',
                     'segment_qty_lag_1m', 'segment_qty_lag_3m', 'segment_qty_lag_6m', 'segment_qty_lag_12m'],
    
    'rolling_features': ['qty_ma_3m', 'qty_ma_6m', 'qty_ma_12m', 'qty_std_3m', 'qty_std_6m', 'qty_std_12m',
                        'qty_growth_1m', 'qty_growth_3m', 'qty_growth_12m'],
    
    'weather_features': ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Max Temp', 'Min Temp',
                        'avg_temp_lead_3m', 'avg_humidity_lead_3m', 'hot_day', 'cold_day', 
                        'humid_day', 'dry_day', 'cdd', 'hdd', 'comfort_index', 'temp_range'],
    
    'product_features': ['inverter_penetration', 'tonnage_1_5_share', 'star_5_share', 
                        'product_age_months', 'adoption_velocity', 'inverter_ratio'],
    
    'geographic_features': ['branch_seasonal_strength', 'branch_market_share', 'branch_performance_ratio',
                           'branch_cluster', 'branch_temp_sensitivity', 'branch_humidity_sensitivity',
                           'inverter_preference'],
    
    'customer_trends': ['Interest', 'interest_lead_3m', 'is_high_interest', 'is_low_interest',
                       'interest_momentum', 'interest_acceleration', 'interest_impact'],
    
    'interaction_features': ['temp_inverter_interaction', 'humidity_tonnage_interaction', 
                            'interest_temp_interaction', 'interest_inverter_interaction',
                            'weather_comfort_score', 'product_attractiveness', 'market_demand_index']
}

# Count features in each group
print("\nFeature Engineering Summary:")
print("=" * 50)
total_features = 0
for group, features in feature_groups.items():
    available_features = [f for f in features if f in monthly_df.columns]
    print(f"{group.upper()}: {len(available_features)} features")
    total_features += len(available_features)

print(f"\nTOTAL ENGINEERED FEATURES: {total_features}")
print(f"Dataset shape: {monthly_df.shape}")
print(f"Date range: {monthly_df['Date'].min()} to {monthly_df['Date'].max()}")
print(f"Branches: {sorted(monthly_df['Branch'].unique())}")
print(f"Segments: {sorted(monthly_df['Segment'].unique())}")

# Save the engineered dataset
output_path = '../data/engineered_dataset.csv'
monthly_df.to_csv(output_path, index=False)
print(f"\nEngineered dataset saved to: {output_path}")

# Display sample of the final dataset
print("\nSample of engineered dataset:")
print(monthly_df[['Date', 'Branch', 'Segment', 'Rating', 'Tonnage', 'Qty', 
                  'Avg Temp', 'Interest', 'inverter_penetration', 'is_peak_month']].head(10))


In [ ]:
# Step 12: Feature Engineering Validation and Recommendations
print("Feature Engineering Validation and Model Preparation Recommendations")
print("=" * 70)

# Validate key EDA insights implementation
print("\n1. EDA INSIGHTS VALIDATION:")
print("-" * 30)

# Check seasonality features
peak_months = monthly_df[monthly_df['is_peak_month'] == 1]['month'].unique()
low_months = monthly_df[monthly_df['is_low_month'] == 1]['month'].unique()
print(f"✅ Peak months implemented: {sorted(peak_months)} (Expected: [2, 3, 12])")
print(f"✅ Low months implemented: {sorted(low_months)} (Expected: [7, 8, 11])")

# Check lag features
lag_features = [col for col in monthly_df.columns if 'lag' in col and 'qty' in col]
print(f"✅ Lag features created: {len(lag_features)} features")
print(f"   - Product-level: {len([f for f in lag_features if not f.startswith('branch') and not f.startswith('segment')])}")
print(f"   - Branch-level: {len([f for f in lag_features if f.startswith('branch')])}")
print(f"   - Segment-level: {len([f for f in lag_features if f.startswith('segment')])}")

# Check weather features with lead
weather_lead_features = [col for col in monthly_df.columns if 'lead_3m' in col]
print(f"✅ Weather features with 3-month lead: {len(weather_lead_features)} features")

# Check product evolution features
product_features = [col for col in monthly_df.columns if any(x in col for x in ['inverter', 'penetration', 'adoption', 'age'])]
print(f"✅ Product evolution features: {len(product_features)} features")

# Check branch heterogeneity features
branch_features = [col for col in monthly_df.columns if 'branch' in col]
print(f"✅ Branch-specific features: {len(branch_features)} features")

print("\n2. MODEL PREPARATION RECOMMENDATIONS:")
print("-" * 40)

print("📊 DATA PREPROCESSING:")
print("   - ✅ Log transformation applied to quantity (log_qty)")
print("   - ✅ Outlier flags created (is_qty_outlier, is_temp_outlier)")
print("   - ✅ Missing values handled with forward-fill and zero-fill")
print("   - ✅ Categorical variables encoded (one-hot and ordinal)")

print("\n🎯 FEATURE SELECTION PRIORITY:")
print("   HIGH PRIORITY:")
print("   - Weather features with 3-month lead (avg_temp_lead_3m, avg_humidity_lead_3m)")
print("   - Seasonal features (is_peak_month, is_low_month, month_sin/cos)")
print("   - Lag features (qty_lag_1m, qty_lag_3m, qty_lag_12m)")
print("   - Product evolution (inverter_penetration, adoption_velocity)")

print("   MEDIUM PRIORITY:")
print("   - Branch-specific patterns (branch_seasonal_strength, branch_market_share)")
print("   - Rolling statistics (qty_ma_3m, qty_ma_6m, qty_std_3m)")
print("   - Weather extremes (hot_day, humid_day, cdd)")

print("   LOW PRIORITY:")
print("   - Customer trends (weak correlation, not statistically significant)")
print("   - Complex interaction features (test for overfitting)")

print("\n🔧 VALIDATION STRATEGY:")
print("   - ✅ Time series split with expanding window")
print("   - ✅ Seasonal validation (ensure complete seasonal cycles)")
print("   - ✅ Branch-wise stratification for geographic robustness")
print("   - ✅ Cross-validation with temporal ordering preserved")

print("\n🎯 TARGET VARIABLE OPTIONS:")
print("   - Primary: log_qty (log-transformed quantity)")
print("   - Alternative: sqrt_qty (square root transformed)")
print("   - Classification: is_peak_month, is_qty_outlier")

print("\n📈 EXPECTED MODEL PERFORMANCE:")
print("   - Strong seasonality patterns should be well captured")
print("   - Weather-sales relationships with 3-month lead should improve predictions")
print("   - Branch heterogeneity requires location-aware modeling")
print("   - Product evolution trends should enhance long-term forecasting")

print("\n🚀 NEXT STEPS:")
print("   1. Run feature selection to identify most predictive features")
print("   2. Implement time series cross-validation")
print("   3. Test multiple algorithms (XGBoost, LightGBM, Prophet, ARIMA)")
print("   4. Create ensemble models combining different approaches")
print("   5. Validate branch-specific vs global models")

print(f"\n✅ FEATURE ENGINEERING COMPLETE!")
print(f"   Total features created: {len(monthly_df.columns)}")
print(f"   Dataset ready for modeling: {monthly_df.shape}")
print(f"   Saved to: ../data/engineered_dataset.csv")
